# 01. 악보로 — 준비된 피아노 연주를 MIDI로

이 노트북은 **준비된 피아노 연주**를 AI가 읽어, 다시 다룰 수 있는 **MIDI (악보 데이터)** 로 바꾸는 데모입니다.
수업에서는 아래의 예시 피아노 음원을 사용해 안정적으로 시연합니다.

**도구**: [Basic Pitch](https://basicpitch.spotify.com/) — Spotify가 만든 오픈소스 채보 AI

▶ Colab에서 실행한다면 아래 설치 셀부터 시작하세요.
서버 시연 환경에서 이미 `setup.sh`를 실행했다면, 설치 셀은 건너뛰고 다음 셀로 바로 넘어가도 됩니다.


In [ ]:
# Colab용 설치 (서버 시연 환경에서는 보통 건너뛰어도 됩니다)
!apt-get -qq install -y fluidsynth > /dev/null 2>&1
!pip install -q basic-pitch librosa soundfile midi2audio matplotlib pretty_midi

import warnings
warnings.filterwarnings('ignore')

print('✅ 설치 완료!')

---
## 1. 준비된 피아노 예시 선택

수업에서는 먼저 **이미 준비된 피아노 음원**으로 데모를 진행합니다.
아래 셀은 먼저 repo의 `assets/` 폴더를 확인하고, 없으면 예시 파일을 다운로드합니다.


In [ ]:
import os
import urllib.request
from pathlib import Path
import IPython.display as ipd

REPO_URL = 'https://raw.githubusercontent.com/joonhyungbae/pianokit/main/assets'
ASSETS_DIR = Path('assets')

examples = {
    'piano_chopin.wav': '쇼팽 스타일 피아노 (서정적, 데모용 기본값)',
    'piano_jazz.wav': '재즈 피아노 즉흥',
    'piano_simple.wav': '짧고 단순한 피아노 프레이즈',
}

print('🎹 사용 가능한 준비된 피아노 예시')
resolved = {}
for fname, desc in examples.items():
    local_asset = ASSETS_DIR / fname
    local_copy = Path(fname)
    if local_asset.exists():
        resolved[fname] = str(local_asset)
        print(f'  ✅ {fname}: assets 폴더에서 사용')
    elif local_copy.exists():
        resolved[fname] = str(local_copy)
        print(f'  ✅ {fname}: 현재 디렉터리에서 사용')
    else:
        try:
            urllib.request.urlretrieve(f'{REPO_URL}/{fname}', fname)
            resolved[fname] = fname
            print(f'  ✅ {fname}: 다운로드 완료')
        except Exception:
            print(f'  ⚠️ {fname}: 준비 실패')
            continue
    print(f'  - {desc}')

input_audio = resolved.get('piano_chopin.wav')

if input_audio and os.path.exists(input_audio):
    print()
    print(f'🎵 현재 선택된 피아노 음원: {input_audio}')
    ipd.display(ipd.Audio(input_audio))
else:
    raise FileNotFoundError('예시 피아노 음원을 찾지 못했습니다. assets 폴더를 확인해주세요.')


---
## 2. AI로 음표 인식하기

Basic Pitch는 오디오의 파형을 분석해서 어떤 음이 언제, 얼마나 세게 연주되었는지를
자동으로 알아냅니다. 사람이 귀로 듣고 악보를 쓰는 과정을 AI가 빠르게 수행하는 셈입니다.

▶ 아래 셀을 실행하세요.

In [ ]:
from basic_pitch.inference import predict

print('🔄 AI가 음표를 분석하고 있습니다...')
model_output, midi_data, note_events = predict(input_audio)

# MIDI 파일 저장
output_midi = 'output.mid'
midi_data.write(output_midi)

print(f'✅ 변환 완료! {len(note_events)}개의 음표를 찾았습니다.')
print(f'💾 MIDI 파일 저장됨: {output_midi}')

---
## 3. 결과 확인 — 피아노롤과 MIDI

아래는 AI가 인식한 음표를 시각화한 **피아노롤**입니다.
- 가로축: 시간 (초)
- 세로축: 음높이
- 색상: 세기

아래에서 변환된 MIDI를 다시 소리로 들어보고, 원본 연주와 짧게 비교해보세요.
이 단계에서 연주는 다시 읽고 편집할 수 있는 **악보 데이터**가 됩니다.

In [ ]:
import pretty_midi
import matplotlib.pyplot as plt
import numpy as np
from midi2audio import FluidSynth
import IPython.display as ipd

# 피아노롤 시각화
pm = pretty_midi.PrettyMIDI(output_midi)

fig, ax = plt.subplots(figsize=(14, 5))

for instrument in pm.instruments:
    for note in instrument.notes:
        ax.barh(
            note.pitch,
            note.end - note.start,
            left=note.start,
            height=0.8,
            color=plt.cm.viridis(note.velocity / 127),
            alpha=0.8
        )

# y축: 음이름 표시 (C4, D4 등)
all_pitches = [n.pitch for inst in pm.instruments for n in inst.notes]
if all_pitches:
    p_min, p_max = min(all_pitches) - 2, max(all_pitches) + 2
    ax.set_ylim(p_min, p_max)
    tick_pitches = [p for p in range(p_min, p_max + 1) if p % 12 == 0]
    ax.set_yticks(tick_pitches)
    ax.set_yticklabels([f'C{p // 12 - 1}' for p in tick_pitches])

ax.set_xlabel('시간 (초)', fontsize=12)
ax.set_ylabel('음높이', fontsize=12)
ax.set_title('🎹 AI가 인식한 피아노롤', fontsize=14)
ax.grid(axis='both', alpha=0.3)
plt.tight_layout()
plt.show()

# MIDI → 오디오 변환
print('\n🔊 MIDI를 소리로 변환 중...')
midi_audio = 'output_midi_audio.wav'
try:
    fs = FluidSynth()
    fs.midi_to_audio(output_midi, midi_audio)
    print('\n🎵 변환된 MIDI 소리:')
    ipd.display(ipd.Audio(midi_audio))
except:
    print('⚠️ MIDI 오디오 변환에 실패했습니다. FluidSynth 사운드폰트가 없을 수 있습니다.')

if os.path.exists(input_audio):
    print('\n🎵 원본 연주:')
    ipd.display(ipd.Audio(input_audio))

---
## 4. 결과 저장

▶ 아래 셀은 생성된 MIDI 파일의 위치를 보여주고, Colab이라면 바로 다운로드합니다.

💡 이 MIDI는 이후 `무대화`나 `협업형` 데모에서 구조를 이해하고 연결할 때 참고할 수 있습니다.


In [ ]:
from pathlib import Path

midi_path = Path(output_midi)
print(f'✅ 결과 파일: {midi_path.resolve()}')

try:
    from google.colab import files
    files.download(str(midi_path))
    print('✅ Colab 다운로드를 시작했습니다.')
except Exception:
    print('ℹ️ 서버/로컬 환경에서는 위 경로의 MIDI 파일을 그대로 사용하면 됩니다.')
